# 🚀 ASTRA — ST-GCN++ Action Recognition Training

**Project**: ASTRA (Astronaut Space Task Recognition & Assistance)

**Model**: ST-GCN++ (Spatial-Temporal Graph Convolutional Network++)

**Classes**: `idle`, `reach`, `pick`, `move`, `place`, `release`

**Dataset**: `synthetic_v3` — Continuous episodes with temporal alignment augmentation

---

## Workflow
1. Setup environment & detect GPU
2. Load dataset (from repo or uploaded ZIP)
3. Verify dataset integrity
4. Train ST-GCN++ with mixed precision
5. Evaluate on train/val/test/hard-test
6. Run continuous rolling-window test
7. Package checkpoint for local download

## Part 1 — Environment Setup

In [ ]:
import sys
import platform
import torch
import numpy as np

print("=" * 60)
print("ASTRA — Environment Info")
print("=" * 60)
print(f"Python:       {sys.version}")
print(f"Platform:     {platform.system()} {platform.release()}")
print(f"PyTorch:      {torch.__version__}")
try:
    import torchvision
    print(f"TorchVision:  {torchvision.__version__}")
except ImportError:
    print("TorchVision:  Not installed")
print(f"NumPy:        {np.__version__}")
print(f"CUDA avail:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"GPU count:    {torch.cuda.device_count()}")
    print(f"GPU memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("GPU:          None (CPU mode)")
print("=" * 60)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()  # Mixed precision only on GPU
print(f"\nUsing device: {DEVICE}")
print(f"Mixed precision: {USE_AMP}")

In [ ]:
# Install additional dependencies if needed
!pip install scikit-learn matplotlib -q

## Part 2 — Data Access

Choose **one** of two methods:

### Method A: Clone Git Repository
```python
!git clone https://github.com/YOUR_ORG/ASTRA.git
%cd ASTRA
```

### Method B: Upload ZIP
Upload `astra_dataset_v3.zip` using the file browser or the cell below.

In [ ]:
#@title Data Access Mode {display-mode: "form"}
DATA_MODE = "upload"  #@param ["git", "upload", "drive"]
GIT_REPO = ""  #@param {type:"string"}
DRIVE_PATH = "/content/drive/MyDrive/ASTRA/data/action_v3"  #@param {type:"string"}

import os

if DATA_MODE == "git" and GIT_REPO:
    if not os.path.exists("ASTRA"):
        !git clone {GIT_REPO}
    %cd ASTRA
    DATA_DIR = "data/action_v3"

elif DATA_MODE == "upload":
    from google.colab import files
    print("Upload astra_dataset_v3.zip...")
    !mkdir -p data/action_v3
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.zip'):
            !unzip -o "{fname}" -d .
    DATA_DIR = "data/action_v3"

elif DATA_MODE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = DRIVE_PATH

print(f"\nData directory: {DATA_DIR}")
print(f"Contents: {os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else 'NOT FOUND'}")

## Part 3 — Dataset Verification

In [ ]:
import json
import hashlib

ACTION_CLASSES = {"idle": 0, "reach": 1, "pick": 2, "move": 3, "place": 4, "release": 5}
CLASS_NAMES = list(ACTION_CLASSES.keys())
NUM_CLASSES = len(ACTION_CLASSES)

# Load splits
splits = {}
for split in ['train', 'val', 'test']:
    path = os.path.join(DATA_DIR, split, 'samples.npz')
    if not os.path.exists(path):
        print(f"ERROR: {path} not found!")
        continue
    d = np.load(path)
    splits[split] = {'X': d['sequences'], 'y': d['labels']}
    print(f"[{split.upper():5s}] Sequences: {d['sequences'].shape} | Labels: {d['labels'].shape}")
    print(f"         Distribution: {dict(zip(CLASS_NAMES, [int((d['labels']==i).sum()) for i in range(NUM_CLASSES)]))}")

# Check hard test
hard_path = os.path.join(DATA_DIR, 'synthetic_hard_test', 'samples.npz')
if os.path.exists(hard_path):
    d = np.load(hard_path)
    splits['hard_test'] = {'X': d['sequences'], 'y': d['labels']}
    print(f"[HARD ] Sequences: {d['sequences'].shape} | Labels: {d['labels'].shape}")

# Leakage check
print("\n--- Leakage Check ---")
hashes = {}
for split in ['train', 'val', 'test']:
    if split not in splits:
        continue
    hashes[split] = set()
    for seq in splits[split]['X']:
        hashes[split].add(hashlib.md5(seq.tobytes()).hexdigest())

if 'train' in hashes and 'val' in hashes:
    print(f"Train∩Val:  {len(hashes['train'] & hashes['val'])} duplicates")
if 'train' in hashes and 'test' in hashes:
    print(f"Train∩Test: {len(hashes['train'] & hashes['test'])} duplicates")
if 'val' in hashes and 'test' in hashes:
    print(f"Val∩Test:   {len(hashes['val'] & hashes['test'])} duplicates")

# Distribution check
print("\n--- Distribution Check ---")
for split in ['train', 'val', 'test']:
    if split not in splits:
        continue
    X = splits[split]['X']
    print(f"[{split.upper():5s}] Mean: {X.mean():.4f} | Std: {X.std():.4f} | Min: {X.min():.4f} | Max: {X.max():.4f}")

## Part 4 — Model Definition

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ---- Spatial Graph ----
def get_spatial_graph(num_joints=17):
    edges = [
        (0, 1), (0, 2), (1, 3), (2, 4),
        (0, 5), (0, 6), (5, 7), (7, 9), (6, 8), (8, 10),
        (5, 11), (6, 12), (11, 12),
        (11, 13), (13, 15), (12, 14), (14, 16),
    ]
    self_loops = [(i, i) for i in range(num_joints)]
    all_edges = edges + self_loops
    A = np.zeros((num_joints, num_joints), dtype=np.float32)
    for i, j in all_edges:
        A[i, j] = 1; A[j, i] = 1
    D = np.sum(A, axis=1)
    D_inv = np.where(D > 0, 1.0 / np.sqrt(D), 0)
    return D_inv[:, None] * A * D_inv[None, :]

A_MATRIX = get_spatial_graph(17)

# ---- Model Blocks ----
class SpatialGraphConv(nn.Module):
    def __init__(self, in_ch, out_ch, A):
        super().__init__()
        self.A = nn.Parameter(torch.from_numpy(A).float(), requires_grad=False)
        self.conv = nn.Conv2d(in_ch, out_ch, 1)
        self.bn = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        x_a = torch.einsum('nctv,vw->nctw', x, self.A)
        return self.bn(self.conv(x_a))

class TemporalConv(nn.Module):
    def __init__(self, ch, ks=9):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, (ks, 1), padding=((ks-1)//2, 0))
        self.bn = nn.BatchNorm2d(ch)
    def forward(self, x):
        return self.bn(self.conv(x))

class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, stride=1, dropout=0.1):
        super().__init__()
        self.gcn = SpatialGraphConv(in_ch, out_ch, A)
        self.tcn = TemporalConv(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(dropout)
        self.residual = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1), nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch else nn.Identity()
        self.pool = nn.AvgPool2d((stride, 1)) if stride > 1 else nn.Identity()

    def forward(self, x):
        res = self.residual(x)
        x = self.relu(self.gcn(x))
        x = self.drop(self.tcn(x))
        x = self.pool(x); res = self.pool(res)
        return self.relu(x + res)

class STGCNPlusPlus(nn.Module):
    def __init__(self, in_channels=3, num_classes=6, num_joints=17, A=None, hidden_dim=64, num_layers=10, dropout=0.1):
        super().__init__()
        self.data_bn = nn.BatchNorm1d(in_channels * num_joints)
        layers = []
        ch = in_channels
        for i in range(num_layers):
            out_ch = hidden_dim * (2 ** min(i // 3, 2))
            stride = 2 if i in [3, 6] else 1
            layers.append(STGCNBlock(ch, out_ch, A, stride, dropout))
            ch = out_ch
        self.layers = nn.Sequential(*layers)
        self.fc = nn.Linear(ch, num_classes)

    def forward(self, x):
        N, C, T, V = x.shape
        x_bn = x.permute(0, 3, 1, 2).contiguous().view(N, V * C, T)
        x_bn = self.data_bn(x_bn)
        x = x_bn.view(N, V, C, T).permute(0, 2, 3, 1).contiguous()
        x = self.layers(x)
        # Spatial pooling
        x = x.mean(dim=-1)  # (N, C, T)
        # Temporal pooling over the final 15 positions
        pool_frames = min(15, x.size(-1))
        x = x[:, :, -pool_frames:].mean(dim=-1)  # (N, C)
        return self.fc(x)

# ---- Dataset ----
class SkeletonActionDataset(Dataset):
    def __init__(self, data_dir):
        data = np.load(os.path.join(data_dir, "samples.npz"))
        self.sequences = torch.from_numpy(data["sequences"]).float()  # (N, T, J, C)
        self.labels = torch.from_numpy(data["labels"]).long()
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx):
        return self.sequences[idx].permute(2, 0, 1), self.labels[idx]  # (C, T, V)

# ---- Build model ----
model = STGCNPlusPlus(
    in_channels=3, num_classes=NUM_CLASSES, num_joints=17,
    A=A_MATRIX, hidden_dim=64, num_layers=10, dropout=0.15
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {n_params:,} total | {n_trainable:,} trainable")
print(f"Model: STGCNPlusPlus")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

## Part 5 — Training Configuration

In [ ]:
#@title Training Hyperparameters {display-mode: "form"}
EPOCHS = 80  #@param {type:"integer"}
BATCH_SIZE = 64  #@param {type:"integer"}
LR = 0.01  #@param {type:"number"}
WEIGHT_DECAY = 5e-4  #@param {type:"number"}
LABEL_SMOOTHING = 0.1  #@param {type:"number"}
EARLY_STOP_PATIENCE = 15  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# DataLoaders
num_workers = 2 if torch.cuda.is_available() else 0
train_ds = SkeletonActionDataset(os.path.join(DATA_DIR, 'train'))
val_ds   = SkeletonActionDataset(os.path.join(DATA_DIR, 'val'))
test_ds  = SkeletonActionDataset(os.path.join(DATA_DIR, 'test'))

# Calculate class weights for WeightedRandomSampler
train_labels = [train_ds[i][1].item() for i in range(len(train_ds))]
class_counts = np.bincount(train_labels)
class_weights = 1.0 / np.maximum(class_counts, 1)  # avoid div by zero
sample_weights = [class_weights[label] for label in train_labels]
sampler = torch.utils.data.WeightedRandomSampler(weights=sample_weights, num_samples=len(train_ds), replacement=True)

print(f"Class counts in train: {class_counts}")
print(f"Class weights: {[round(w, 4) for w in class_weights]}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=num_workers, drop_last=True, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Batch size: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LR}")
print(f"Weight decay: {WEIGHT_DECAY} | Label smoothing: {LABEL_SMOOTHING}")
print(f"Early stopping patience: {EARLY_STOP_PATIENCE}")
print(f"AMP (mixed precision): {USE_AMP}")

In [ ]:
# Sanity check: verify balanced sampling
print("Running sanity check on one batch...")
batch_x, batch_y = next(iter(train_loader))
batch_counts = np.bincount(batch_y.numpy(), minlength=NUM_CLASSES)
print(f"Batch size: {batch_x.size(0)}")
print(f"Class counts in batch: {batch_counts}")
for i, count in enumerate(batch_counts):
    print(f"  {CLASS_NAMES[i]:7s}: {count:2d} ({(count/batch_x.size(0))*100:.1f}%)")
print("Sampling is balanced if distribution is roughly equal (~16.6% each).")

## Part 6 — Training Loop

In [ ]:
import time
from datetime import datetime
import torch.optim as optim

optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

# Training state
OUTPUT_DIR = "checkpoints/action/STGCNPP-ASTRA-v3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

best_val_acc = 0.0
best_epoch = 0
patience_counter = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

t_start = time.time()
print(f"Training started at {datetime.now().strftime('%H:%M:%S')}")
print("=" * 80)

for epoch in range(EPOCHS):
    # ---- Train ----
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(batch_x)
            loss = criterion(logits, batch_y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * batch_x.size(0)
        _, preds = logits.max(dim=1)
        train_correct += (preds == batch_y).sum().item()
        train_total += batch_x.size(0)

    scheduler.step()
    train_loss /= max(train_total, 1)
    train_acc = train_correct / max(train_total, 1)

    # ---- Validate ----
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
            val_loss += loss.item() * batch_x.size(0)
            _, preds = logits.max(dim=1)
            val_correct += (preds == batch_y).sum().item()
            val_total += batch_x.size(0)

    val_loss /= max(val_total, 1)
    val_acc = val_correct / max(val_total, 1)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    lr = scheduler.get_last_lr()[0]
    marker = ""

    # Best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        patience_counter = 0
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "val_acc": val_acc,
            "val_loss": val_loss,
            "classes": CLASS_NAMES,
            "num_classes": NUM_CLASSES,
        }, os.path.join(OUTPUT_DIR, "best.pt"))
        marker = f" ★ NEW BEST"
    else:
        patience_counter += 1

    # Periodic checkpoint
    if (epoch + 1) % 10 == 0:
        torch.save({"model": model.state_dict(), "epoch": epoch},
                   os.path.join(OUTPUT_DIR, f"epoch_{epoch+1}.pt"))

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
          f"loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | "
          f"lr={lr:.6f}{marker}")

    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\n⛔ Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
        break

# Save last checkpoint
torch.save({
    "model": model.state_dict(),
    "epoch": epoch,
    "val_acc": val_acc,
    "classes": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
}, os.path.join(OUTPUT_DIR, "last.pt"))

elapsed = time.time() - t_start
print(f"\n{'=' * 80}")
print(f"Training complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Best val accuracy: {best_val_acc:.4f} at epoch {best_epoch}")
print(f"Train/Val gap: {history['train_acc'][-1] - history['val_acc'][-1]:.4f}")

## Part 7 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Curves'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()
print(f"Saved to {OUTPUT_DIR}/training_curves.png")

## Part 8 — Three-Way Evaluation + Hard Test

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Load best checkpoint
best_ckpt = torch.load(os.path.join(OUTPUT_DIR, 'best.pt'), map_location=DEVICE)
model.load_state_dict(best_ckpt['model'])
model.eval()
print(f"Loaded best checkpoint from epoch {best_ckpt['epoch']+1} (val_acc={best_ckpt['val_acc']:.4f})")

def evaluate_split(split_name, data_dir):
    path = os.path.join(data_dir, 'samples.npz')
    if not os.path.exists(path):
        print(f"[{split_name}] Not found: {path}")
        return None, None
    data = np.load(path)
    X, y_true = data['sequences'], data['labels']

    y_pred = []
    y_probs = []
    for seq in X:
        tensor = torch.from_numpy(seq).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(tensor)
            probs = torch.softmax(logits, dim=-1).cpu().numpy().squeeze()
        y_pred.append(int(np.argmax(probs)))
        y_probs.append(probs)

    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')

    # Top-5
    top5 = np.mean([1 if y_true[i] in np.argsort(y_probs[i])[-5:] else 0 for i in range(len(y_true))])

    print(f"\n{'=' * 50}")
    print(f"[{split_name.upper()}]  Acc: {acc:.4f} | F1: {f1:.4f} | Top-5: {top5:.4f}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    print("Confusion Matrix:")
    print(cm)

    return acc, f1

results = {}
for split in ['train', 'val', 'test']:
    acc, f1 = evaluate_split(split, os.path.join(DATA_DIR, split))
    if acc is not None:
        results[split] = {'accuracy': acc, 'f1': f1}

# Hard test
hard_dir = os.path.join(DATA_DIR, 'synthetic_hard_test')
if os.path.exists(hard_dir):
    acc, f1 = evaluate_split('hard_test', hard_dir)
    if acc is not None:
        results['hard_test'] = {'accuracy': acc, 'f1': f1}

# Summary table
print(f"\n{'=' * 60}")
print(f"{'Split':12s} | {'Accuracy':>10s} | {'Macro F1':>10s} | {'Gap':>10s}")
print(f"{'-'*12}-+-{'-'*10}-+-{'-'*10}-+-{'-'*10}")
train_acc = results.get('train', {}).get('accuracy', 0)
for split in ['train', 'val', 'test', 'hard_test']:
    if split in results:
        gap = train_acc - results[split]['accuracy']
        print(f"{split:12s} | {results[split]['accuracy']:10.4f} | {results[split]['f1']:10.4f} | {gap:+10.4f}")

## Part 9 — Confusion Matrix Visualization

In [ ]:
# Visualize confusion matrix for val set
data = np.load(os.path.join(DATA_DIR, 'val', 'samples.npz'))
X_val, y_val = data['sequences'], data['labels']

y_pred_val = []
for seq in X_val:
    tensor = torch.from_numpy(seq).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        y_pred_val.append(int(torch.argmax(logits, dim=-1).item()))

cm = confusion_matrix(y_val, y_pred_val, labels=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Validation Confusion Matrix (Acc: {accuracy_score(y_val, y_pred_val):.3f})')

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.colorbar(im); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## Part 10 — Continuous Rolling-Window Test

In [ ]:
from collections import deque, Counter

class ActionSmoother:
    def __init__(self, window_size=15, min_count=8):
        self.window = deque(maxlen=window_size)
        self.min_count = min_count
        self.current = "idle"
    def update(self, action):
        self.window.append(action)
        if len(self.window) >= self.min_count:
            counts = Counter(self.window)
            top = counts.most_common(1)[0]
            if top[1] >= self.min_count:
                self.current = top[0]
        return self.current

# Generate a held-out continuous episode
# Inline minimal episode generator (no external imports needed)
def generate_test_episode(rng):
    """Generate one continuous IDLE->REACH->PICK->MOVE->PLACE->RELEASE->IDLE episode."""
    cx = 0.5
    base = np.zeros((17, 2), dtype=np.float32)
    base[0] = [cx, 0.15]; base[1] = [cx-0.02, 0.13]; base[2] = [cx+0.02, 0.13]
    base[3] = [cx-0.04, 0.15]; base[4] = [cx+0.04, 0.15]
    base[5] = [cx-0.10, 0.25]; base[6] = [cx+0.10, 0.25]
    base[7] = [cx-0.175, 0.35]; base[8] = [cx+0.175, 0.35]
    base[9] = [cx-0.21, 0.45]; base[10] = [cx+0.21, 0.45]
    base[11] = [cx-0.08, 0.50]; base[12] = [cx+0.08, 0.50]
    base[13] = [cx-0.08, 0.65]; base[14] = [cx+0.08, 0.65]
    base[15] = [cx-0.08, 0.80]; base[16] = [cx+0.08, 0.80]

    obj_pos = np.array([0.72, 0.48]); tgt_pos = np.array([0.28, 0.47])
    wrist, elbow = 10, 8
    rest = base[wrist].copy()

    actions = [
        ('idle',    rng.randint(15, 40)),
        ('reach',   rng.randint(15, 35)),
        ('pick',    rng.randint(10, 20)),
        ('move',    rng.randint(20, 45)),
        ('place',   rng.randint(10, 30)),
        ('release', rng.randint(8, 18)),
        ('idle',    rng.randint(15, 40)),
    ]
    total = sum(d for _, d in actions)
    frames = np.zeros((total, 17, 3), dtype=np.float32)
    labels = np.zeros(total, dtype=np.int64)

    t = 0
    for action, dur in actions:
        cid = ACTION_CLASSES[action]
        for i in range(dur):
            p = i / max(dur-1, 1)
            sk = base.copy() + rng.randn(17, 2) * 0.003
            if action == 'reach':
                sk[wrist] = rest + (obj_pos - rest) * p
            elif action == 'pick':
                sk[wrist] = obj_pos + np.array([0, -0.06*p])
            elif action == 'move':
                sk[wrist] = obj_pos + (tgt_pos - obj_pos) * p + np.array([0, -0.05])
            elif action == 'place':
                sk[wrist] = tgt_pos + np.array([0, -0.05*(1-p)])
            elif action == 'release':
                sk[wrist] = tgt_pos + (rest - tgt_pos) * p
            sk[elbow] = base[elbow] + (sk[wrist] - base[wrist]) * 0.5
            conf = np.ones((17, 1), dtype=np.float32) * 0.9
            frames[t, :, :2] = sk; frames[t, :, 2:] = conf
            labels[t] = cid; t += 1
    return frames[:t], labels[:t], actions

# Run the rolling test
rng = np.random.RandomState(777)
ep_frames, ep_labels, ep_actions = generate_test_episode(rng)
T = len(ep_frames)

print(f"Episode: {T} frames")
print(f"Action sequence: {[f'{a}({d})' for a, d in ep_actions]}")
print("=" * 80)

smoother = ActionSmoother(window_size=15, min_count=8)
skeleton_buf = deque(maxlen=60)
last_smoothed = 'idle'
transitions_detected = []
frame_results = []

for t in range(T):
    skeleton_buf.append(ep_frames[t])
    raw_pred, conf = 'idle', 0.0

    if len(skeleton_buf) >= 16:
        seq = np.array(list(skeleton_buf))
        padded = np.zeros((60, 17, 3), dtype=np.float32)
        L = min(len(seq), 60)
        padded[:L] = seq[:L]
        for i in range(L, 60): padded[i] = seq[-1]

        tensor = torch.from_numpy(padded).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(tensor)
            probs = torch.softmax(logits, dim=-1).cpu().numpy().squeeze()
        idx = int(np.argmax(probs))
        raw_pred = CLASS_NAMES[idx]; conf = probs[idx]

    smoothed = smoother.update(raw_pred)
    frame_results.append((t, CLASS_NAMES[ep_labels[t]], raw_pred, conf, smoothed))

    if smoothed != last_smoothed:
        transitions_detected.append((t, last_smoothed, smoothed, CLASS_NAMES[ep_labels[t]]))
        last_smoothed = smoothed

    if t % 20 == 0:
        print(f"Frame {t:3d} | True: {CLASS_NAMES[ep_labels[t]]:7s} | Raw: {raw_pred:7s} ({conf:.2f}) | Smoothed: {smoothed:7s}")

print(f"\n{'=' * 80}")
print("DETECTED TRANSITIONS:")
for t, frm, to, true in transitions_detected:
    match = '✓' if to == true else '✗'
    print(f"  Frame {t:3d}: {frm} → {to}  (true: {true}) {match}")

# Transition accuracy
expected = ['idle', 'reach', 'pick', 'move', 'place', 'release', 'idle']
detected_seq = ['idle'] + [to for _, _, to, _ in transitions_detected]
print(f"\nExpected sequence:  {expected}")
print(f"Detected sequence:  {detected_seq}")

correct = sum(1 for e, d in zip(expected, detected_seq) if e == d)
print(f"Sequence accuracy: {correct}/{len(expected)} = {correct/len(expected):.2%}")

## Part 11 — Save Metadata & Package Checkpoint

In [ ]:
import subprocess

# Git commit
try:
    git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
except:
    git_commit = 'unknown'

# Training metadata
metadata = {
    "model": "STGCNPP",
    "version": "ASTRA-v3",
    "dataset": "synthetic_v3",
    "classes": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
    "input_shape": [60, 17, 3],
    "n_joints": 17,
    "n_frames": 60,
    "epochs_run": len(history['train_loss']),
    "epochs_max": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "dropout": 0.15,
    "optimizer": "SGD",
    "scheduler": "CosineAnnealing",
    "seed": SEED,
    "device": str(DEVICE),
    "mixed_precision": USE_AMP,
    "n_train": len(train_ds),
    "n_val": len(val_ds),
    "n_test": len(test_ds),
    "best_val_acc": best_val_acc,
    "best_epoch": best_epoch,
    "training_time_seconds": elapsed,
    "results": results,
    "history": history,
    "python_version": sys.version,
    "pytorch_version": torch.__version__,
    "cuda_version": str(torch.version.cuda) if torch.cuda.is_available() else 'N/A',
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    "git_commit": git_commit,
    "timestamp": datetime.now().isoformat(),
}

with open(os.path.join(OUTPUT_DIR, 'training_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

# Class mapping
with open(os.path.join(OUTPUT_DIR, 'class_mapping.json'), 'w') as f:
    json.dump(ACTION_CLASSES, f, indent=2)

# Metrics summary
with open(os.path.join(OUTPUT_DIR, 'metrics.json'), 'w') as f:
    json.dump(results, f, indent=2)

print("Saved: training_metadata.json, class_mapping.json, metrics.json")
print(f"\nCheckpoint directory: {OUTPUT_DIR}")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

In [ ]:
# Package into downloadable ZIP
import shutil

zip_name = 'STGCNPP-ASTRA-v3'
shutil.make_archive(zip_name, 'zip', '.', OUTPUT_DIR)
print(f"Created: {zip_name}.zip")

# Download (Colab only)
try:
    from google.colab import files
    files.download(f'{zip_name}.zip')
    print("Download triggered!")
except:
    print(f"Not in Colab — manually download: {zip_name}.zip")

## Part 12 — Anti-Overfitting Analysis

In [ ]:
# Overfitting analysis
final_train_acc = history['train_acc'][-1]
final_val_acc = history['val_acc'][-1]
gap = final_train_acc - final_val_acc

print("=" * 60)
print("OVERFITTING ANALYSIS")
print("=" * 60)
print(f"Final Train Acc:  {final_train_acc:.4f}")
print(f"Final Val Acc:    {final_val_acc:.4f}")
print(f"Best Val Acc:     {best_val_acc:.4f}")
print(f"Train-Val Gap:    {gap:.4f}")
print()

if gap > 0.3:
    print("⚠️  SEVERE OVERFITTING (gap > 0.3)")
    print("   Recommendations:")
    print("   - Increase dropout")
    print("   - Increase weight decay")
    print("   - Increase dataset diversity")
    print("   - Add more augmentation")
elif gap > 0.15:
    print("⚠️  MODERATE OVERFITTING (gap > 0.15)")
    print("   Consider more regularization or data augmentation.")
elif gap > 0.05:
    print("✓  MILD OVERFITTING (gap 0.05-0.15) — acceptable for synthetic data.")
else:
    print("✓  MINIMAL OVERFITTING — good generalization.")

print(f"\nDECISION: {'PROCEED to GRU' if best_val_acc > 0.4 and gap < 0.4 else 'FIX ST-GCN first'}")